# 11. Functional readers from Perturb-seq: dominance within a line and switching between lines (Fig. 5d-g)

For each core motif and each candidate reader TF with a ChIP-seq experiment and a knockdown in the genome-wide Perturb-seq screens of **K562** (Replogle et al. 2022) and **HEK293T** (X-Atlas/Orion), the knockout effect on the motif's ChIP-defined target genes is compared across readers: within a line (panels **d**, **e**: the leader's advantage over the other readers) and between the two lines (**f**, **g**: the log2 K562/HEK293T ratio of a reader's effect, against 300 control TFs that read no core motif).

Inputs (Zenodo `fig5_reader_switch_inputs.tar.gz` under `$ENHANCER_ROOT`): `perturbseq_qn/` holds the **quantile-normalised knockout-effect vectors** of both screens on their 8,236 shared genes, which is what every step below reads (the raw screens, 2.9 GB, are needed only to rebuild these vectors and are not distributed), the shared motif loci of notebook 10, and the feasibility table; Zenodo `fig4_fig5_motif_sites_on_peaks.tar.gz` holds the motif sites in K562 and HEK293 peaks; the ENCODE peaks of K562 and HEK293 come from the download of notebook 08 (`01_prepare_peaks.py` there also built `data/{K562,HEK293}/`).

Chain (`code/chipseq_enrichment/`, argument `top90chip` = readers defined by their top-90 % peaks): `09e_reader_binding_pctl.py` (reader target genes) -> `09k` (reader curves, panel d) -> `09q` (effective readers, panel e) -> `09x` (cross-line control TFs) -> `09z`, `09z3` (switch pairs, panel f) -> `09z5` (switch magnitude per gene, panel g) -> `09k4b_fig5_defg_allreaders.py` (the sheet). Every step loads the effect vectors and the K562/HEK293 peak scans through the loader prefix of `08c_alltargets_stats.py`, which reads `perturbseq_qn/` when it is present and otherwise falls back to the raw screens, which are not distributed; the whole chain takes about 45 min on a CPU.

**Panel f draws every reader of a motif** as its own cross-line curve (adopted 2026-10-18), in place of the single widest reciprocal reader pair per motif; `09k4_fig5_defg.py`, which draws the earlier version, is still in `code/` as provenance.

In [ ]:
# ---- Shared setup (identical in every notebook) --------------------------------------------------------
import os, sys, json, subprocess, time, glob, shutil
REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))          # notebooks/ lives one level below the repository root
CODE = os.path.join(REPO, "code")
roots = json.load(open(os.path.join(REPO, ".roots.json")))          # written by bin/rewrite_paths.py (see PATHS.md)
ENHANCER_ROOT, SEQ_ROOT = roots["enhancer_root"], roots["seq_root"]
def run(cmd, cwd=None, env=None):
    """Run one repository script from the right folder and stream its output here. Stops the notebook if it fails."""
    print("$", cmd, "   (in", cwd or os.getcwd(), ")"); t = time.time()
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, executable="/bin/bash", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in p.stdout: print(line, end="")
    p.wait(); print(f"[exit {p.returncode}, {time.time()-t:.0f} s]")
    assert p.returncode == 0, "this step failed - read the output above"
def n_motifs(meme):
    """Number of MOTIF entries in a MEME file."""
    return sum(1 for l in open(meme) if l.startswith("MOTIF")) if os.path.exists(meme) else None
def show_logos(meme, n=5, title=""):
    """Sequence logos of the first n motifs of a MEME file (logomaker)."""
    import numpy as np, pandas as pd, matplotlib.pyplot as plt, logomaker
    blocks = open(meme).read().split("MOTIF ")[1:n+1]; fig, axes = plt.subplots(1, len(blocks), figsize=(2.2*len(blocks), 1.4))
    for ax, b in zip(np.atleast_1d(axes), blocks):
        rows = [l.split() for l in b.split("\n")[2:] if l.strip() and not l.startswith("letter") and not l.startswith("URL")]
        pwm = pd.DataFrame([[float(x) for x in r] for r in rows if len(r) == 4], columns=list("ACGT"))
        logomaker.Logo(logomaker.transform_matrix(pwm, from_type="probability", to_type="information"), ax=ax); ax.set_title(b.split("\n")[0], fontsize=8); ax.set_yticks([]); ax.set_xticks([])
    fig.suptitle(title, fontsize=9); plt.tight_layout(); plt.show()
print("repository:", REPO); print("ENHANCER_ROOT:", ENHANCER_ROOT); print("SEQ_ROOT:", SEQ_ROOT)


In [ ]:
ENCODE = f"{ENHANCER_ROOT}/encode4_tf_chipseq"     # bin/download_encode_peaks.py --dest $ENHANCER_ROOT/encode4_tf_chipseq
def encode_status(cells):
    for c in cells:
        n = len(glob.glob(f"{ENCODE}/{c}/*/*.bed")); want = sum(1 for l in open(f"{REPO}/data/encode/encode_tf_chipseq_peaks.tsv") if l.split("\t")[0] == c)
        print(f"  {'ok ' if n == want else '-- '} {c}: {n} of {want} ENCODE peak files")
CE = f"{CODE}/chipseq_enrichment"; RS = f"{CE}/reader_switch"; TAG = "top90chip"; WORK = f"{ENHANCER_ROOT}/notebook_runs/fig5"; os.makedirs(WORK, exist_ok=True)
import numpy as np, pandas as pd
from IPython.display import Image, display
encode_status(["K562", "HEK293"])
NEED = {"quantile-normalised K562 effects": (f"{ENHANCER_ROOT}/perturbseq_qn/perturbseq_qn_K562.npz", "fig5_reader_switch_inputs.tar.gz"),
        "quantile-normalised HEK293T effects": (f"{ENHANCER_ROOT}/perturbseq_qn/perturbseq_qn_HEK293T.npz", "fig5_reader_switch_inputs.tar.gz"),
        "motif sites in K562 peaks": (f"{CE}/results/fimo/K562/fimo_peaks/fimo.tsv", "fig4_fig5_motif_sites_on_peaks.tar.gz"),
        "motif sites in HEK293 peaks": (f"{CE}/results/fimo/HEK293/fimo_peaks/fimo.tsv", "fig4_fig5_motif_sites_on_peaks.tar.gz"),
        "shared loci": (f"{RS}/out/shared_loci.parquet", "fig5_reader_switch_inputs.tar.gz"),
        "feasibility table": (f"{CE}/results/reader_switching/hek_k562_switch_feasibility.csv", "fig5_reader_switch_inputs.tar.gz"),
        "CAGE TSS": (f"{ENHANCER_ROOT}/IGV_tracks/CAGE_TSS_hg38.bed", "ships in the repository, linked by bin/setup_work_dir.py")}
missing = []
for k, (p, where) in NEED.items():
    ok = os.path.exists(p)
    print(f"  {'ok ' if ok else '-- '} {k}: {p}")
    if not ok: missing.append(f"    {k}\n      expected at {p}\n      comes from {where}")
assert not missing, ("this notebook cannot run without:\n" + "\n".join(missing) +
                     "\n\n  Unpack the archive(s) named above into $ENHANCER_ROOT and re-run this cell."
                     "\n  Every step below reads the quantile-normalised effect vectors; the raw Perturb-seq"
                     "\n  screens are NOT distributed and are not needed.")
if not os.path.exists(f"{CE}/data/HEK293/tf_peak_sets.pkl"): run("python 01_prepare_peaks.py K562 HEK293", cwd=CE)     # notebook 08 normally did this
print(open(f"{ENHANCER_ROOT}/perturbseq_qn/README_perturbseq_qn.txt").read())
shutil.copy(f"{CE}/results/prof_plan/dominant_reader_{TAG}.csv", f"{WORK}/dominant_reader_{TAG}.published.csv")

## 11.1 The effect vectors

In [ ]:
Q = np.load(f"{ENHANCER_ROOT}/perturbseq_qn/perturbseq_qn_K562.npz", allow_pickle=True); print("K562:", Q["delta_qn"].shape, "(perturbed genes x shared genes), effective knockdowns:", int(Q["eff_kd"].sum()), "| responsive-target floor", float(Q["thr"]))
i = list(Q["tfs"]).index("GATA1"); v = Q["delta_qn"][i]; print("GATA1 knockdown in K562: strongest effects on", ", ".join(Q["sym_ensg"][np.isin(Q["sym_ensg"], Q["genes"][np.argsort(-np.abs(np.nan_to_num(v)))[:6]])][:6]))

## 11.2 Reader target genes from the ENCODE peaks (minutes)
`09e_reader_binding_pctl.py`: for K562 and HEK293, every candidate reader's peaks at or above its 10th within-TF percentile, intersected with the shared loci -> `reader_switch/out/reader_binding_top90.parquet`.

In [ ]:
shutil.copy(f"{RS}/out/reader_binding_top90.parquet", f"{WORK}/reader_binding_top90.published.parquet")
run("python 09e_reader_binding_pctl.py", cwd=CE)
a = pd.read_parquet(f"{RS}/out/reader_binding_top90.parquet"); b = pd.read_parquet(f"{WORK}/reader_binding_top90.published.parquet"); print(f"{len(a):,} (cell, gene, motif, reader) records; published {len(b):,}")

## 11.3 The analysis chain (about 40 min)
Each script prints what it loads and writes; results go to `results/prof_plan/` (tables, and `prof_plan/fig_reader_curves/` for the figures).

In [ ]:
for s in ("09k_cobound_reader_curves", "09q_effective_readers", "09x_cross_line_control", "09z_switch_pair_figures", "09z3_figure_ready", "09z5_switch_magnitude_pergene"):
    run(f"python {s}.py {TAG}", cwd=CE)
d = pd.read_csv(f"{CE}/results/prof_plan/dominant_reader_{TAG}.csv"); old = pd.read_csv(f"{WORK}/dominant_reader_{TAG}.published.csv")
print(d.iloc[:, :8].round(3).to_string(index=False)); print("\nsame panels and leaders as published:", d.iloc[:, :3].astype(str).equals(old.iloc[:, :3].astype(str)))
print(pd.read_csv(f"{CE}/results/prof_plan/switch_figready_pairs_{TAG}.csv").iloc[:, :6].round(3).to_string(index=False))

## 11.4 Panels d-g

In [ ]:
run(f"python 09k4b_fig5_defg_allreaders.py {TAG}", cwd=CE)          # the paper's Fig. 5d-g
display(Image(f"{CE}/results/prof_plan/fig_reader_curves/fig5_defg_allreaders_{TAG}.png", width=900))